# 21 · Native expressions versus Python UDFs

**Scenario:** Compare equivalent calculations and inspect execution-plan differences.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. Python UDF availability depends on runtime and access mode. No pandas UDFs are used.

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Prefer Spark-native expressions

Spark can optimize native expressions. A Python UDF may introduce serialization boundaries and version/environment requirements.


In [ ]:
valid=df.filter('distance > 0 AND fare IS NOT NULL')
native=valid.withColumn('native_value',F.round(F.col('fare')/F.col('distance'),2))
native.select('fare','distance','native_value').show(5)
native.explain(mode='formatted')


**Expected check:** The plan contains the native arithmetic expression.


## Step 3 — Run a small scalar UDF comparison

This UDF does no I/O, uses no pandas, and is defined only to compare semantics. Use a tolerance because rounding implementations can differ at exact half values.


In [ ]:
@F.udf(returnType=T.DoubleType())
def ratio(fare,distance):
    if fare is None or distance is None or distance<=0: return None
    return float(fare/distance)
compared=valid.withColumn('python_ratio',ratio('fare','distance')).withColumn('native_ratio',F.col('fare')/F.col('distance'))
assert compared.filter(F.abs(F.col('python_ratio')-F.col('native_ratio'))>1e-9).count()==0
compared.explain(mode='formatted')


**Expected check:** The unrounded ratios agree; the plan exposes the Python evaluation boundary.


## Practice

Replace a Python string-cleaning UDF with trim, lower, regexp_replace and higher-order functions.

## Solution guide

Build Column expressions and compare output using exceptAll. Use UDFs only when native functionality cannot express the required operation.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
